[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/Chapter_Colabs/blob/main/Colab_H.ipynb)


# Set H — BMI‑1: micro:bit Only (LEGO–Colab)
**Author: Neural Engineering Laboratory, University of Missouri - Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair

> Local runtime required for serial access. Provides a FEAR command bridge to micro:bit over USB UART.

---
## Table of Contents
- [H0 Starter](#h0)
- [H1 FEAR activation logic (host)](#h1)
- [H2 micro:bit firmware (flash as main.py)](#h2)
- [Reflection](#reflection)
---


### H0 — Starter (local runtime only)
**Idea.** Initialize the USB serial connection to the micro:bit; detect the board; open a port; expose a minimal function (`send_fear()`) that sends an ASCII "FEAR " command. This cell forms the “physical I/O layer” for the BMI bridge.

**What to change.**
*   `VID_MICROBIT`, `PID_MICROBIT` if using different microcontroller versions.
*   `BAUD` if using alternate firmware.
*   Increase serial timeout if the device enumerates slowly.

**Sanity checks.**
*   Running H0 should print “H0 ready.”
*   If the device is not detected, run `list_ports.comports()` to check enumeration.
*   Verify that `open_serial()` returns a live port and that `send_fear()` writes without error.

**Predict → verify.**
*   USB enumeration time affects the first command latency; subsequent commands should be faster.
*   Changing BAUD on one side only should break communication—use this to reason about handshake requirements.

**Exercises.**
1.  **Modify H0** to print the detected port name; verify it matches your OS device list.
2.  **Add a helper** that prints the round trip time for a dummy command (e.g., "PING").
3.  **Inspect the effect** of different serial timeouts (100–500 ms) on robustness.

In [ ]:

!pip -q install pyserial
import time, sys, serial, serial.tools.list_ports as list_ports
VID_MICROBIT=3368; PID_MICROBIT=516; BAUD=115200
_ser=None

def find_port():
    for p in list_ports.comports():
        if getattr(p,'vid',None)==VID_MICROBIT and getattr(p,'pid',None)==PID_MICROBIT:
            return p.device
    for p in list_ports.comports():
        d=(p.description or '').lower()
        if any(x in d for x in ['micro:bit','mbed','daplink','bbc']): return p.device
    return None

def open_serial():
    global _ser
    if _ser and _ser.is_open: return _ser
    port=find_port()
    if not port: raise RuntimeError('micro:bit not found')
    _ser=serial.Serial(port,BAUD,timeout=0.2); time.sleep(0.2); return _ser

def send_fear():
    s=open_serial(); s.write(b'FEAR
'); s.flush()

print('H0 ready.')


### H1 — FEAR activation logic (host side)
**Idea.** Convert a scalar internal state (e.g., fear probability from Set G) into a device trigger using a threshold rule and a cooldown to avoid rapid re-firing. The widget UI allows interactive testing.

**What to change.**
*   `thr.value` (default threshold 0.7); adjust based on your chosen behavioral scoring.
*   `cd_ms`: cooldown in milliseconds.
*   Integrate your actual model output by calling `on_fear(prob)` where $prob \in [0,1]$.

**Sanity checks.**
*   Manual “Test FEAR” click should send exactly one command per press.
*   Triggering via model output should obey both threshold and cooldown constraints.
*   Inspect printed timestamps to confirm correct cooldown enforcement.

**Predict → verify.**
*   Lowering the threshold increases trigger frequency; raising it reduces sensitivity.
*   Cooldown prevents “event storms” if the internal state fluctuates around the threshold.
*   Fast-varying inputs from Set G may require smoothing or hysteresis.

**Exercises.**
1.  **Sweep the threshold** (0.4 $\rightarrow$ 0.9) and document how often FEAR is triggered for a given trace.
2.  **Replace thresholding** with a hysteresis window (e.g., 0.65/0.75) and compare stability.
3.  **Add logging** of (timestamp, prob, triggered?) for 50 trials to analyze jitter.

In [ ]:

import ipywidgets as w
from IPython.display import display

thr=w.FloatSlider(value=0.7,min=0,max=1,step=0.01,description='FEAR thr')
b=w.Button(description='Test FEAR',button_style='danger')

def on_fear(prob,thr=0.7,cd_ms=1500):
    if not hasattr(on_fear,'t0'): on_fear.t0=0
    now=time.time()*1000
    if prob>=thr and now-on_fear.t0>cd_ms:
        send_fear(); on_fear.t0=now; print('FEAR sent',prob)

def _t(_): send_fear()
b.on_click(_t)
display(w.VBox([thr,b]))


### H2 — micro:bit firmware (flash as main.py)
**Idea.** Implement a simple device-side event loop that listens for "FEAR" over UART and executes a legible action (LED skull + tone sequence). This is the “expression” half of the BMI bridge.

**What to change.**
*   Modify the `fear()` function to customize the action (patterns, durations, sounds).
*   Extend the protocol to accept additional commands (e.g., "CALM", "GO").
*   Change UART baud rate if needed (must match H0).

**Sanity checks.**
*   After flashing, the device should clear its display and respond only when "FEAR" is received.
*   Confirm that the LED image appears and the tone sequence plays identically each time.
*   Test debouncing by sending "FEAR" twice rapidly.

**Predict → verify.**
*   If host sends commands faster than `fear()` completes, device responsiveness drops; buffering may occur.
*   Adding a device-side cooldown avoids overwriting an ongoing action.

**Exercises.**
1.  **Add a second command** ("CALM") that displays a neutral icon; update the host to test it.
2.  **Log the time** between host send and device action start (use micro:bit `running_time()`); compute latency distribution.
3.  **Implement a graded encoding** (e.g., "LEVEL 0", "LEVEL 1", …) and map it to brightness or tone frequency.


In [ ]:

from microbit import *
import music, uart
uart.init(baudrate=115200)

def fear():
    display.show(Image.SKULL)
    music.play(['c5:1','b4:1','a4:2'])
    display.clear()

while True:
    if uart.any():
        if uart.readline().strip().upper()==b'FEAR': fear()
    sleep(10)


### Reflection
**Idea.** Reason about the mapping from internal circuit dynamics $\rightarrow$ stable scalar state $\rightarrow$ threshold/hysteresis rule $\rightarrow$ device action, maintaining the verification mindset from Sets A–G.

**Exercises / Prompts.**
1.  **Define your internal state** and justify its behavioral relevance.
2.  **Propose a transparent conversion rule** (threshold or hysteresis) and defend its interpretability.
3.  **Measure latency** from state crossing in the model to device action; identify jitter sources.
4.  **Identify one communication error mode** and propose a mitigation (timeouts, checksums, acknowledgments).
5.  **Design a verification test** that sends deterministic sequences and checks device responses.
6.  **Explain why** educational BMIs benefit from explicit, minimal protocols instead of opaque ML pipelines.
7.  **Extend the contract** to handle graded states or multi-effector outputs.



---

## Practice / Discussion Questions — Set H — Translation to BMI

1) Define an **internal model state** to export to a device and why it’s meaningful.
2) *Design*: Choose a **threshold rule** to convert a scalar state into a trigger; justify.
3) How would you **measure latency** from state crossing to device action?
4) One **communication error mode** and its mitigation.
5) An **encoding** for graded states (levels/PWM) and why it helps.
6) *Predict*: What mismatch occurs if internal events are faster than the effector; fix?
7) Why emphasize **transparent I/O semantics** in educational BMIs?
8) Create a **simple interface contract** and a verification test.
9) Two sources of **timing jitter** to check if actions are inconsistent.
10) How to **log events** to timestamp internal and external actions.
11) One reason to **debounce** or smooth state transitions before triggering.
12) How to document **assumptions** so others can reproduce your mapping.
13) One **ethical/interpretive** caution mapping state → action.
14) *Extend*: Add a second effector representing a different property.
15) Propose a **stress test** and pass/fail criteria.
16) Why a BMI lab is a strong **capstone** for model→behavior.
17) One **accessibility** consideration in signal→action mapping.
18) How to **simulate** device logic before hardware.
19) *Compare*: Direct thresholding vs **hysteresis window**—which and why?
20) How Set H reinforces **mechanistic understanding** rather than replacing it.
